# Test Notebook for baseline.py

This notebook tests the timbre transfer functions from `src/baseline.py`.
It uses a voice as the source and a violin as the target, similar to `playground.ipynb`.

In [ ]:
import numpy as np
import soundfile as sf
import pyworld as pw
import librosa
import matplotlib.pyplot as plt
import librosa.display
from IPython.display import Audio, display
from pathlib import Path
import sys

# Add src to path to import baseline
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import baseline

print(f"Project root: {PROJECT_ROOT}")

# Define paths
source_audio_path = PROJECT_ROOT / "data/VocalSet/FULL/female1/long_tones/forte/f1_long_forte_a.wav"
target_audio_path = PROJECT_ROOT / "data/150_Gm_BllywdVln_SP_39_01.wav"
output_dir = PROJECT_ROOT / "data" / "baseline_outputs"
output_dir.mkdir(exist_ok=True)

# Display source and target audio
print("Source audio (voice):")
display(Audio(filename=source_audio_path))
print("Target audio (violin):")
display(Audio(filename=target_audio_path))

## 1. Load and Preprocess Audio
Load audio files and extract WORLD parameters (F0, spectral envelope, aperiodicity).

In [ ]:
# Load audio
y_src, sr = baseline.load_audio(source_audio_path)
y_tgt, _ = baseline.load_audio(target_audio_path, sr=sr)

# Ensure target is long enough
if len(y_tgt) < len(y_src):
    y_tgt = np.pad(y_tgt, (0, len(y_src) - len(y_tgt)), 'wrap')
else:
    y_tgt = y_tgt[:len(y_src)]

# Convert to float64 for WORLD
y_src = y_src.astype(np.float64)
y_tgt = y_tgt.astype(np.float64)

# Decompose using WORLD
f0_src, sp_src, ap_src = baseline.world_decompose(y_src, sr)
f0_tgt, sp_tgt, ap_tgt = baseline.world_decompose(y_tgt, sr)

## 2. Test Source-Filter Vocoder Timbre Transfer
This method uses the source's excitation (F0 and aperiodicity) and the target's spectral envelope.

In [ ]:
# We want to apply the target's timbre to the source's pitch.
# The function in baseline.py is source_filter_transfer(f0_tgt, ap_tgt, sp_src)
# which does the opposite. Let's test the intended version first.

# Intended: source F0/ap, target sp
y_sf_intended = baseline.world_synthesize(f0_src, sp_tgt, ap_src, sr)
output_path = output_dir / "source_filter_intended.wav"
baseline.save_audio(output_path, y_sf_intended, sr)

print("Source-Filter Transfer (Source F0, Target SP):")
display(Audio(filename=output_path))

# As written in baseline.py: target F0/ap, source sp
y_sf_as_written = baseline.source_filter_transfer(f0_tgt, ap_tgt, sp_src)
output_path = output_dir / "source_filter_as_written.wav"
baseline.save_audio(output_path, y_sf_as_written, sr)

print("Source-Filter Transfer (Target F0, Source SP):")
display(Audio(filename=output_path))

## 3. Test Spectral Envelope Transfer
This method transfers the spectral envelope from the source to the target.

In [ ]:
sp_transferred = baseline.spectral_envelope_transfer(sp_src, sp_tgt)
y_spectral_env = baseline.world_synthesize(f0_src, sp_transferred, ap_src, sr)
output_path = output_dir / "spectral_envelope_transfer.wav"
baseline.save_audio(output_path, y_spectral_env, sr)

print("Spectral Envelope Transfer:")
display(Audio(filename=output_path))

## 4. Test Phase Vocoder-Based Timbre Modification
This method combines the magnitude of the source's STFT with the phase of the target's STFT.

In [ ]:
y_phase_voc = baseline.phase_vocoder_transfer(y_src, y_tgt, sr)
output_path = output_dir / "phase_vocoder_transfer.wav"
baseline.save_audio(output_path, y_phase_voc, sr)

print("Phase Vocoder Transfer:")
display(Audio(filename=output_path))

## 5. Visualization of Results
Let's look at the spectrograms of the original and transferred audio.

In [ ]:
def plot_spectrograms(audios, titles, sr):
    fig, ax = plt.subplots(nrows=len(audios), ncols=1, sharex=True, figsize=(10, 3 * len(audios)))
    for i, (y, title) in enumerate(zip(audios, titles)):
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        librosa.display.specshow(D, y_axis='log', x_axis='time', sr=sr, ax=ax[i])
        ax[i].set_title(title)
        ax[i].set_ylabel('Frequency [Hz]')
    ax[-1].set_xlabel('Time [s]')
    plt.tight_layout()
    plt.show()

audios_to_plot = [y_src, y_tgt, y_sf_intended, y_spectral_env, y_phase_voc]
titles = [
    'Original Source (Voice)',
    'Original Target (Violin)',
    'Source-Filter Transfer',
    'Spectral Envelope Transfer',
    'Phase Vocoder Transfer'
]

plot_spectrograms(audios_to_plot, titles, sr)